# Yaya-125M — Kaggle Training

**Before running:** Settings → Accelerator → **GPU T4 x2** (or P100) → Save

## Session flow
| Session | What happens |
|---------|-------------|
| 1st | Tokenize Wikipedia (~15 min) + pretrain 5000 steps (~10 hrs) |
| 2nd+ | Download checkpoint dataset → auto-resume pretrain or start SFT |

## Checkpoint persistence between sessions
Kaggle doesn't have persistent storage like Drive. Between sessions:
1. After each session ends, go to **Output tab** → download `yaya-checkpoints.zip`
2. Upload it as a **Kaggle Dataset** (once)
3. On the next session, attach that dataset → checkpoint is auto-detected

Or skip persistence and just train start-to-finish in one ~12h session (T4 x2 is faster).


## Step 0 — Check GPU & setup

In [ ]:
import torch, os, sys, glob, shutil

if torch.cuda.is_available():
    name = torch.cuda.get_device_name(0)
    vram = round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1)
    print(f'GPU : {name}  ({vram} GB VRAM)')
    print(f'GPU count: {torch.cuda.device_count()}')
    print(f'PyTorch: {torch.__version__}')
    DTYPE = 'bfloat16' if torch.cuda.is_bf16_supported() else 'float16'
    IS_A100 = 'A100' in name
    IS_DUAL = torch.cuda.device_count() >= 2
else:
    raise RuntimeError('No GPU found. Go to Settings → Accelerator → GPU T4 x2')

print(f'dtype: {DTYPE}  |  A100: {IS_A100}  |  dual GPU: {IS_DUAL}')

# Working dirs
WORK_DIR     = '/kaggle/working'
CKPT_DIR     = f'{WORK_DIR}/yaya-checkpoints'
PRETRAIN_DIR = f'{CKPT_DIR}/pretrain'
SFT_DIR      = f'{CKPT_DIR}/sft'
DATA_CACHE   = f'{WORK_DIR}/data_cache'
WIKI_TRAIN   = f'{DATA_CACHE}/wiki_train'
WIKI_EVAL    = f'{DATA_CACHE}/wiki_eval'

for d in [PRETRAIN_DIR, SFT_DIR, DATA_CACHE, WIKI_TRAIN, WIKI_EVAL]:
    os.makedirs(d, exist_ok=True)

print('Dirs ready.')

In [ ]:
import subprocess

# Clone or update repo
if not os.path.exists('/kaggle/working/miss-yaya'):
    subprocess.run(['git', 'clone', 'https://github.com/jaylink-coder/miss-yaya.git',
                    '/kaggle/working/miss-yaya'], check=True)
else:
    subprocess.run(['git', '-C', '/kaggle/working/miss-yaya', 'pull'], check=True)

# Install deps
subprocess.run(['pip', 'install', '-q', 'sentencepiece', 'pyyaml', 'datasets'], check=True)

os.chdir('/kaggle/working/miss-yaya/yaya-ai')
sys.path.insert(0, '/kaggle/working/miss-yaya/yaya-ai')
print('Ready:', os.getcwd())

# Restore checkpoint from attached Kaggle dataset (if any)
# Attach your saved dataset as /kaggle/input/yaya-checkpoints before running
ATTACHED_CKPT = '/kaggle/input/yaya-checkpoints'
if os.path.exists(ATTACHED_CKPT):
    print(f'Restoring checkpoints from attached dataset: {ATTACHED_CKPT}')
    shutil.copytree(ATTACHED_CKPT, CKPT_DIR, dirs_exist_ok=True)
    pretrain_ckpts = sorted(glob.glob(f'{PRETRAIN_DIR}/checkpoint-*'))
    sft_ckpts = sorted(glob.glob(f'{SFT_DIR}/checkpoint-*'))
    print(f'  Pretrain checkpoints: {len(pretrain_ckpts)}')
    print(f'  SFT checkpoints: {len(sft_ckpts)}')
else:
    print('No attached checkpoint dataset — starting fresh.')

## Phase 1 — Pretrain on Wikipedia

Gives Yaya world knowledge. **Skip if PRETRAIN_DIR already has checkpoints** (auto-detected).

In [ ]:
import yaml

# Download + tokenize Wikipedia
if not os.path.exists(f'{WIKI_TRAIN}/shard_00000.bin'):
    print('Downloading + tokenizing Wikipedia 20220301.en (20% sample)...')
    from datasets import load_dataset
    from src.tokenizer.tokenizer import YayaTokenizer
    import numpy as np

    # wikimedia/wikipedia is the script-free Parquet version
    ds = load_dataset('wikimedia/wikipedia', '20220301.en', split='train[:20%]')
    print(f'Loaded {len(ds):,} articles')

    tok = YayaTokenizer('data/tokenizer/yaya_tokenizer.model')
    split = ds.train_test_split(test_size=0.005, seed=42)

    def tokenize_and_save(dataset, out_dir, fname):
        tokens = []
        for i, row in enumerate(dataset):
            tokens.extend(tok.encode(row['text']))
            if (i + 1) % 10000 == 0:
                print(f'  {i+1:,} articles, {len(tokens)/1e6:.0f}M tokens')
        arr = np.array(tokens, dtype=np.uint16)
        arr.tofile(f'{out_dir}/{fname}')
        print(f'  Saved {len(arr)/1e6:.0f}M tokens → {out_dir}/{fname}')

    tokenize_and_save(split['train'], WIKI_TRAIN, 'shard_00000.bin')
    tokenize_and_save(split['test'],  WIKI_EVAL,  'eval.bin')
else:
    size_mb = os.path.getsize(f'{WIKI_TRAIN}/shard_00000.bin') / 1e6
    print(f'Wikipedia already cached ({size_mb:.0f} MB) — skipping download')

# Patch pretrain config
with open('configs/training/train_125m.yaml') as f:
    pt_cfg = yaml.safe_load(f)

pt_cfg['checkpointing']['save_dir']              = PRETRAIN_DIR
pt_cfg['training']['dtype']                      = DTYPE
pt_cfg['training']['gradient_checkpointing']     = True
pt_cfg['training']['max_steps']                  = 5000
pt_cfg['training']['max_seq_length']             = 1024
pt_cfg['training']['per_device_batch_size']      = 8 if IS_A100 else 4
pt_cfg['training']['gradient_accumulation_steps'] = 4 if IS_A100 else 8
pt_cfg['data']['train_data'] = WIKI_TRAIN
pt_cfg['data']['eval_data']  = WIKI_EVAL

with open('configs/training/train_125m.yaml', 'w') as f:
    yaml.dump(pt_cfg, f)

pretrain_ckpts = sorted(glob.glob(f'{PRETRAIN_DIR}/checkpoint-*'))
if pretrain_ckpts:
    resume = f'--resume {pretrain_ckpts[-1]}'
    print(f'Resuming pretrain from: {pretrain_ckpts[-1]}')
else:
    resume = ''
    print('Starting pretrain from scratch')

print(f'batch={pt_cfg["training"]["per_device_batch_size"]}, '
      f'accum={pt_cfg["training"]["gradient_accumulation_steps"]}, dtype={DTYPE}')

In [ ]:
!WANDB_DISABLED=true python -u scripts/train_sft.py \
    --model_config configs/model/yaya_125m.yaml \
    --train_config configs/training/train_125m.yaml \
    {resume}

## Phase 2 — Download 260k reasoning examples

GSM8K + MetaMath + OpenHermes. **Skip if already downloaded.**

In [ ]:
LOCAL_DATA  = 'data/sft/yaya_reasoning_large.jsonl'
CACHED_DATA = f'{DATA_CACHE}/yaya_reasoning_large.jsonl'

if os.path.exists(CACHED_DATA):
    shutil.copy(CACHED_DATA, LOCAL_DATA)
    import subprocess as sp
    n = int(sp.check_output(['wc', '-l', LOCAL_DATA]).split()[0])
    print(f'Data already cached: {n:,} examples')
else:
    print('Downloading reasoning datasets (GSM8K + MetaMath + OpenHermes)...')
    !python scripts/download_reasoning_data.py
    shutil.copy(LOCAL_DATA, CACHED_DATA)
    print(f'Cached to: {CACHED_DATA}')